In [18]:
import numpy as np
import numba as nb
from pycbc.events.hm_utils import get_indices_jit_2_ifo, get_indices_jit_3_ifo

@nb.jit(nopython=True)
def index_combinations(tlen, t2_coinc_window, t3_coinc_window, 
                       dtype=np.int64):
    """For three detectors, this is equivalent to calling 
    get_indices_jit_3_ifo, but is generally faster. For 
    two detectors this just calls get_indices_jit_2_ifo 
    directly.
    """
    if t3_coinc_window is None:
        return get_indices_jit_2_ifo(tlen, t2_coinc_window, dtype)
    elif 2 * max(t2_coinc_window, t3_coinc_window) > tlen:
        return get_indices_jit_3_ifo(tlen, t2_coinc_window, t3_coinc_window,
                                     dtype)
    # forgetting the tails at first
    largest_window = max(t2_coinc_window, t3_coinc_window)
    idx_1_mid = (
        np.arange(tlen - 2 * (t3_coinc_window + 1), dtype=dtype).repeat(
            (2 * t2_coinc_window + 1) * (2 * t3_coinc_window + 1)
        )
        + largest_window
        + 1
    )
#start
    idx_2_mid = np.empty((tlen - 2 * (t3_coinc_window + 1)) * (2 * t2_coinc_window + 1), dtype=dtype)

    for i in range(2 * t3_coinc_window + 1):
        for j in range(tlen - 2 * (t3_coinc_window + 1)):
            idx_2_mid[i * (tlen - 2 * (t3_coinc_window + 1)) + j] = idx_1_mid[j] + np.arange(-t2_coinc_window, t2_coinc_window + 1, dtype=dtype)[i]


    idx_3_mid = np.empty((len(idx_1_mid) * (2 * t2_coinc_window + 1) * (tlen - 2 * (t3_coinc_window + 1)),), dtype=dtype)
    pos = 0
    for i in range(len(idx_1_mid)):
        for j in range(2 * t3_coinc_window + 1):
            for k in range(-t3_coinc_window, t3_coinc_window + 1):
                idx_3_mid[pos] = idx_1_mid[i] + k
                pos += 1
#end
    idx_1_ends, idx_2_ends, idx_3_ends = get_indices_jit_3_ifo(
        2 * (t3_coinc_window + 1), t2_coinc_window, t3_coinc_window,
        dtype=dtype
    ).T
    # now get the tails
    n_start = int(len(idx_1_ends) / 2)
    idx_1 = np.concatenate(
        (
            idx_1_ends[:n_start],
            idx_1_mid,
            idx_1_ends[-n_start:] + tlen - (2 * t3_coinc_window + 2),
        )
    )
    idx_2 = np.concatenate(
        (
            idx_2_ends[:n_start],
            idx_2_mid,
            idx_2_ends[-n_start:] + tlen - (2 * t3_coinc_window + 2),
        )
    )
    idx_3 = np.concatenate(
        (
            idx_3_ends[:n_start],
            idx_3_mid,
            idx_3_ends[-n_start:] + tlen - (2 * t3_coinc_window + 2),
        )
    )
    return np.array([idx_1, idx_2, idx_3]).T


def index_combinations_old(tlen, t2_coinc_window, t3_coinc_window, 
                       dtype=np.int64):
    """For three detectors, this is equivalent to calling 
    get_indices_jit_3_ifo, but is generally faster. For 
    two detectors this just calls get_indices_jit_2_ifo 
    directly.
    """
    if t3_coinc_window is None:
        return get_indices_jit_2_ifo(tlen, t2_coinc_window, dtype)
    elif 2 * max(t2_coinc_window, t3_coinc_window) > tlen:
        return get_indices_jit_3_ifo(tlen, t2_coinc_window, t3_coinc_window,
                                     dtype)
    # forgetting the tails at first
    largest_window = max(t2_coinc_window, t3_coinc_window)
    idx_1_mid = (
        np.arange(tlen - 2 * (t3_coinc_window + 1), dtype=dtype).repeat(
            (2 * t2_coinc_window + 1) * (2 * t3_coinc_window + 1)
        )
        + largest_window
        + 1
    )
    idx_2_mid = idx_1_mid + np.tile(
        np.arange(-t2_coinc_window, t2_coinc_window + 1, dtype=dtype).repeat(
            2 * t3_coinc_window + 1
        ),
        (tlen - 2 * (t3_coinc_window + 1)),
    )
    idx_3_mid = idx_1_mid + np.tile(
        np.arange(-t3_coinc_window, t3_coinc_window + 1, dtype=dtype),
        (tlen - 2 * (t3_coinc_window + 1)) * (2 * t2_coinc_window + 1),
    )
    idx_1_ends, idx_2_ends, idx_3_ends = get_indices_jit_3_ifo(
        2 * (t3_coinc_window + 1), t2_coinc_window, t3_coinc_window,
        dtype=dtype
    ).T
    # now get the tails
    n_start = int(len(idx_1_ends) / 2)
    idx_1 = np.concatenate(
        (
            idx_1_ends[:n_start],
            idx_1_mid,
            idx_1_ends[-n_start:] + tlen - (2 * t3_coinc_window + 2),
        )
    )
    idx_2 = np.concatenate(
        (
            idx_2_ends[:n_start],
            idx_2_mid,
            idx_2_ends[-n_start:] + tlen - (2 * t3_coinc_window + 2),
        )
    )
    idx_3 = np.concatenate(
        (
            idx_3_ends[:n_start],
            idx_3_mid,
            idx_3_ends[-n_start:] + tlen - (2 * t3_coinc_window + 2),
        )
    )
    return np.array([idx_1, idx_2, idx_3]).T

In [19]:
idx = index_combinations(tlen=3072,t2_coinc_window=8,t3_coinc_window=17)

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mNo implementation of function Function(<built-in function array>) found for signature:
 
 >>> array(list(array(int64, 1d, C))<iv=None>)
 
There are 6 candidate implementations:
[1m      - Of which 4 did not match due to:
      Overload in function '_OverloadWrapper._build.<locals>.ol_generated': File: numba/core/overload_glue.py: Line 131.
        With argument(s): '(list(array(int64, 1d, C))<iv=None>)':[0m
[1m       Rejected as the implementation raised a specific error:
         TypingError: Failed in nopython mode pipeline (step: nopython frontend)
       [1m[1m[1mNo implementation of function Function(<intrinsic stub>) found for signature:
        
        >>> stub(list(array(int64, 1d, C))<iv=None>)
        
       There are 2 candidate implementations:
       [1m  - Of which 2 did not match due to:
         Intrinsic in function 'stub': File: numba/core/overload_glue.py: Line 35.
          